In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
# load_dotenv(dotenv_path=Path(".env"))

print(os.getenv("OPENAI_API_KEY"))
print(os.getenv("GROQ_API_KEY"))

In [ ]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
def llm(prompt):
    response = openai_client.responses.create(
        # model='gpt-5.4-mini',
        model="openai/gpt-oss-120b",
        input=prompt,
    )
    return response.output_text

In [ ]:
question = 'I just discovered the course. Can I join now?'
answer = llm(question)
print(answer)

In [ ]:
context = '''
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram &amp; Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active.

edit on GitHub
#Cloud alternatives with GPU
Check the quota and reset cycle carefully. Is the free hours limit per month or per week? Usually, if you change the configuration, the free hours quota might also be adjusted, or it might be billed separately.

Potential options include:

Google Colab
Kaggle
Databricks (possibly)
Consider using GPTs to discover more options. Be aware that some platforms might have restrictions on what you can and cannot install, so ensure to read what is included in the free vs paid tier.
'''

In [ ]:
prompt = f'''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
'''

In [ ]:
question = 'I just discovered the course. Can I join now?'
answer = llm(prompt)
print(answer)

In [ ]:
import json

# Point directly to the single JSON file
file_path = '/all_documents.json'

with open(file_path, 'r', encoding='utf-8') as f:
    documents = json.load(f)

print(len(documents))

In [ ]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)
index.fit(documents)

In [ ]:
documents[100]['course']

In [ ]:
search_results = index.search(
    question,
    boost_dict={'question': 2.0, 'section': 0.5},
    filter_dict={'course': 'data-engineering'},
    num_results=5
)
search_results

In [ ]:
def search(question, course='llm'):
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict = {'course': course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [ ]:
question

In [ ]:
search_results = search(question, 'data-engineering')
search_results

In [ ]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [ ]:
USER_PROMPT_TEMPALATE = '''
Question:
{question}

Context:
{context}
'''

In [ ]:
search_results

In [ ]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()

In [ ]:
new_context = build_context(search_results)
print(new_context)

In [ ]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPALATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [ ]:
prompt = build_prompt(question, search_results)
print(prompt)

In [ ]:
response = openai_client.responses.create(
    # model='gpt-5.4-mini',
    model="openai/gpt-oss-120b",
    input=prompt
)

In [ ]:
response.output_text

In [ ]:
response.output[0].content[0].text

In [ ]:
response.usage

In [ ]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)
cost

In [ ]:
INSTRUCTIONS

In [ ]:
message_history = [
    {'role': 'developer', 'content': INSTRUCTIONS},
    {'role': 'user', 'content': prompt}
]

response = openai_client.responses.create(
    # model='gpt-5.4-mini',
    model="openai/gpt-oss-120b",
    input=message_history
)

In [ ]:
response.output_text

In [ ]:
# model='gpt-5.4-mini',
def llm(instructions, user_prompt, model="openai/gpt-oss-120b"):
    message_history = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )
    return response.output_text

In [ ]:
# model='gpt-5.4-mini',
def rag(query, model="openai/gpt-oss-120b"):
    search_results = search(query) # in filter course = llm, data-engineering
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [ ]:
answer = rag('How do I get a certificate?')
print(answer)